---
web scraping prompt
---
Check the attached HTML file, then help me make a Python script that uses the beautiful soup library to scrap this website: https://www.scrapethissite.com/pages/simple/ and extract the countries and their info into a clean pandas dataframe that can be saved as a csv file at the end

In [1]:
import pandas as pd 
import plotly.express as px

In [2]:
# ! pip install requests beautifulsoup4 pandas

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

# URL of the target page
url = "https://www.scrapethissite.com/pages/simple/"

# Send an HTTP GET request to the website
response = requests.get(url)
response.raise_for_status()  # Raise an error if the request failed

# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# List to hold the extracted country dictionaries
countries_data = []

# Find all container divs that hold individual country information
country_divs = soup.find_all("div", class_="country")

for country in country_divs:
  # Extract country name (stripping out whitespace/newlines)
  name = country.find("h3", class_="country-name").text.strip()

  # Extract specific fields using their unique classes
  capital = country.find("span", class_="country-capital").text.strip()
  population = country.find("span", class_="country-population").text.strip()
  area = country.find("span", class_="country-area").text.strip()

  # Append the structured data to our list
  countries_data.append({
      "Country": name,
      "Capital": capital,
      "Population": int(population) if population.isdigit() else population,
      "Area (km2)": float(area) if area else None,
  })

# Convert the list of dictionaries into a Pandas DataFrame
df = pd.DataFrame(countries_data)

# Display the first few rows of the DataFrame
print(df.head())

# Save the DataFrame to a CSV file
output_file = r"../RawData/countries_of_the_world.csv"
df.to_csv(output_file, index=False)
print(f"\nData successfully scraped and saved to {output_file}")

In [4]:
Countries = pd.read_csv(r'..\RawData\countries_of_the_world.csv')
Countries

,Country,Capital,Population,Area (km2)
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0
...,...,...,...,...
245,Yemen,Sanaa,23495361,527970.0
246,Mayotte,Mamoudzou,159042,374.0
247,South Africa,Pretoria,49000000,1219912.0
248,Zambia,Lusaka,13460305,752614.0


In [5]:
Countries.sample(5)


,Country,Capital,Population,Area (km2)
139,Montenegro,Podgorica,666730,14026.0
125,Laos,Vientiane,6368162,236800.0
43,Ivory Coast,Yamoussoukro,21058798,322460.0
167,Nepal,Kathmandu,28951852,140800.0
42,Switzerland,Bern,7581000,41290.0


In [6]:
Countries.info()


<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Country     250 non-null    str    
 1   Capital     242 non-null    str    
 2   Population  250 non-null    int64  
 3   Area (km2)  250 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 7.9 KB


In [7]:
print(f"Countries Data have {Countries.shape[0]} Rows & {Countries.shape[1]} Columns With {Countries.nunique().sum()} Unique Values")

Countries Data have 250 Rows & 4 Columns With 984 Unique Values


In [8]:
Countries.duplicated().sum()

np.int64(0)

In [9]:
Countries.isnull().sum()[Countries.isna().sum() > 0]

Capital    8
dtype: int64

In [10]:
Nulls = Countries[Countries['Capital'].isnull() == True]
Nulls

,Country,Capital,Population,Area (km2)
8,Antarctica,NaN,0,14000000.0
33,Bouvet Island,NaN,0,49.0
95,Heard Island and McDonald Islands,NaN,0,412.0
102,Israel,NaN,7353985,20770.0
105,British Indian Ocean Territory,NaN,4000,60.0
182,Palestine,NaN,3800000,5970.0
219,Tokelau,NaN,1466,10.0
231,U.S. Minor Outlying Islands,NaN,0,0.0


In [11]:
Countries.iat[102,0] = "Nothing"
Countries.iat[102,0]

'Nothing'

In [12]:
Countries['Capital'] = Countries['Capital'].fillna('No Capital')
Countries.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Country     250 non-null    str    
 1   Capital     250 non-null    str    
 2   Population  250 non-null    int64  
 3   Area (km2)  250 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 7.9 KB


What are the top 5 countries by population, and what are their corresponding capitals?

In [13]:
Countries.sort_values(by='Population',ascending=False).head(5).reset_index(drop=True).drop(columns='Area (km2)')

,Country,Capital,Population
0,China,Beijing,1330044000
1,India,New Delhi,1173108018
2,United States,Washington,310232863
3,Indonesia,Jakarta,242968342
4,Brazil,Brasília,201103330


Which 5 countries have the smallest geographic area?

In [14]:
Countries[['Country','Area (km2)']].sort_values('Area (km2)').head(5).reset_index(drop=True)

,Country,Area (km2)
0,U.S. Minor Outlying Islands,0.00
1,Vatican City,0.44
2,Monaco,1.95
3,Gibraltar,6.50
4,Tokelau,10.00


What is the population density (Population divided by Area) for each country, and which countries are top 5 density?

In [24]:
Countries['Population_density'] = (Countries['Population'] / Countries['Area (km2)']).round(2)
top_den = Countries.sort_values(by= 'Population_density',ascending=False).head(5).reset_index(drop=True)
top_den

,Country,Capital,Population,Area (km2),Population_density
0,Monaco,Monaco,32965,1.95,16905.13
1,Singapore,Singapore,4701069,692.70,6786.59
2,Hong Kong,Hong Kong,6898686,1092.00,6317.48
3,Gibraltar,Gibraltar,27884,6.50,4289.85
4,Vatican City,Vatican City,921,0.44,2093.18


How many countries have a population greater than 100 million?

In [27]:
pop_100 = Countries[Countries['Population'] > 100000000]
pop_100['Country'].count()

np.int64(11)

Which countries have an area smaller than 1,000 square kilometers (micro-states)?

In [17]:
smallest = Countries[['Country' ,'Area (km2)']][Countries['Area (km2)'] < 1000].sort_values(by='Area (km2)').head(5)
# OR
Countries.groupby('Country')['Area (km2)'].sum().sort_values().reset_index().head(5)

,Country,Area (km2)
0,U.S. Minor Outlying Islands,0.00
1,Vatican City,0.44
2,Monaco,1.95
3,Gibraltar,6.50
4,Tokelau,10.00


What is the global sum and average of the Population and Area (km2) across all 250 rows?

In [18]:
Countries[['Population','Area (km2)']].agg(['sum','mean'])

,Population,Area (km2)
sum,6.861419e+09,1.499092e+08
mean,2.744568e+07,5.996369e+05


In [19]:
summary_stats = Countries[["Population", "Area (km2)"]].agg(["sum", "mean"])
print(summary_stats)

        Population    Area (km2)
sum   6.861419e+09  1.499092e+08
mean  2.744568e+07  5.996369e+05


In [20]:
# اختيار أعلى 10 دول سكانياً
top_pop = Countries.nlargest(10, "Population")

fig = px.bar(
    top_pop,
    x="Country",
    y="Population",
    text="Capital",
    color="Population",
    color_continuous_scale="Viridis",  # تعديل الاسم هنا بالشرطة السفلية
    title="Top 10 Most Populous Countries",
    labels={"Population": "Total Population", "Country": "Country"},
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig.show()

In [21]:

fig = px.scatter(
    smallest,
    x="Area (km2)",
    y="Country",
    size="Area (km2)", # حجم النقطة يعبر عن المساحة
    color="Country",
    hover_name="Country",
    text="Country",
    title="Top 5 Smallest Countries (Scatter Plot)",
    labels={"Area (km2)": "Area in km²", "Country": "Country Name"},
)

# تنسيق العرض
fig.update_traces(textposition="top center", marker=dict(sizemode='diameter', sizeref=0.1, sizemin=15))
fig.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)

fig.show()

In [26]:
fig = px.bar(
    pop_100,
    x="Country",
    y="Population",
    text="Capital",
    color="Population",
    # الطريقة الأكيدة: تحديد الحدود هنا في الـ bar plot مباشرة
    range_color=[100000000, pop_100["Population"].max()], 
    color_continuous_scale="Viridis",
    title="Countries with Population Greater than 100 Million",
    labels={"Population": "Population", "Country": "Country"},
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig.show()

In [25]:
fig = px.bar(
    top_den,
    x='Country',
    y='Population_density',
    text='Capital',
    color='Population_density',
    color_continuous_scale='Magma', # تدرج لوني ممتاز للكثافة
    title='Top 5 Countries by Population Density',
    labels={'Population_density': 'Density (People / km²)', 'Country': 'Country'}
)

fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig.show()